<a href="https://colab.research.google.com/github/Vaishnavi-dhavade20/128597_Opinion_Simulation/blob/main/op4u.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install networkx matplotlib seaborn pandas ipywidgets

In [ ]:
import pandas as pd
import seaborn as sns
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

In [ ]:
N = 50

#Barabasi network gives realistic influencer structure

G = nx.barabasi_albert_graph(N, 2)

#Ensure graph is connected

while not nx.is_connected(G):
  G = nx.barabasi_albert_graph(N, 2)

#Initial Network Visualization

plt.figure(figsize=(8,6))

nx.draw(
G,
with_labels=True,
node_color="lightblue",
node_size=600
)

plt.title("Initial Social Network")
plt.show()

In [ ]:
#Initial random openions
opinions = np.random.rand(N)

print("Initial Opinions:")
print(opinions)

In [ ]:
# Influence matrix with different influence strengths

W = np.zeros((N, N))

for i in G.nodes():

    neighbors = list(G.neighbors(i))

    if len(neighbors) > 0:

        # Generate random influence strengths
        random_weights = np.random.rand(len(neighbors))

        # Normalize so row sum = 1
        random_weights = random_weights / np.sum(random_weights)

        # Assign weights
        for idx, j in enumerate(neighbors):

            W[i][j] = random_weights[idx]

In [ ]:
labels = [f"Person {i}" for i in range(N)]

df_W = pd.DataFrame(W * 100).round(0).astype(int)

df_W.index = labels
df_W.columns = labels

df_W

In [ ]:
plt.figure(figsize=(12,10))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(
G,
pos,
node_color="lightblue",
node_size=700
)


nx.draw_networkx_labels(G, pos)

edges = []
edge_labels = {}

for i in range(N):
  for j in range(N):

    # Show only strong influence connections
    if W[i][j] > 0.25:

        # j influences i
        edges.append((j, i))

        edge_labels[(j, i)] = f"{int(W[i][j]*100)}%"

nx.draw_networkx_edges(
G,
pos,
edgelist=edges,
edge_color="red",
arrows=True,
arrowstyle="->",
arrowsize=20,
width=2,
connectionstyle="arc3,rad=0.15"
)


nx.draw_networkx_edge_labels(
G,
pos,
edge_labels=edge_labels,
font_size=8,
label_pos=0.6,
bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)

plt.title("Directional Influence Network")
plt.axis("off")
plt.show()

In [ ]:
#centrality measure
degree = nx.degree_centrality(G)

betweenness = nx.betweenness_centrality(G)

closeness = nx.closeness_centrality(G)

centrality_methods = {
"Degree": degree,
"Betweenness": betweenness,
"Closeness": closeness
}

In [ ]:
centrality_df = pd.DataFrame({
"Degree": degree,
"Betweenness": betweenness,
"Closeness": closeness
}) * 100

centrality_df = centrality_df.round(2)

print("Centrality Measures (%)")
display(centrality_df)

In [ ]:
#Top Influencers
print("Top Degree Node:",
max(degree, key=degree.get))

print("Top Betweenness Node:",
max(betweenness, key=betweenness.get))

print("Top Closeness Node:",
max(closeness, key=closeness.get))

In [ ]:
pip install ipywidgets


In [ ]:
import ipywidgets as widgets

In [ ]:
#define bot
bot = 1

In [ ]:
node_colors = []

for i in range(N):

    if i == bot:
        node_colors.append("red")

    elif degree[i] > 0.2:
        node_colors.append("yellow")   # High influence

    elif degree[i] > 0.1:
        node_colors.append("green")    # Medium influence

    else:
        node_colors.append("skyblue")  # Low influence

plt.figure(figsize=(9,7))

node_sizes = [500 + 2000 * degree[i] for i in range(N)]
nx.draw(
    G,
    with_labels=True,
    node_color=node_colors,
    node_size=node_sizes
)

legend = [
    mpatches.Patch(color='red', label='Bot'),
    mpatches.Patch(color='yellow', label='High Influence'),
    mpatches.Patch(color='green', label='Medium Influence'),
    mpatches.Patch(color='skyblue', label='Low Influence')
]

plt.legend(handles=legend)
plt.title("Influence Classification Network")
plt.show()

In [ ]:
def plot_centrality(choice):

    plt.figure(figsize=(6,4))

    if choice == "Degree":
        values = list(degree.values())
        title = "Degree Centrality"

    elif choice == "Betweenness":
        values = list(betweenness.values())
        title = "Betweenness Centrality"

    else:
        values = list(closeness.values())
        title = "Closeness Centrality"

    nx.draw(
        G,
        node_color=values,
        cmap=plt.cm.viridis,
        with_labels=True,
        node_size=600
    )

    plt.title(title)
    plt.show()

In [ ]:
dropdown = widgets.Dropdown(
    options=["Degree", "Betweenness", "Closeness"],
    value="Degree",
    description="Centrality:"
)

In [ ]:
widgets.interactive(plot_centrality, choice=dropdown)

In [ ]:
# Store convergence results
convergence_history = {}

# Run convergence simulation for each centrality
for method_name, centrality_values in centrality_methods.items():

    # Fresh opinion state
    opinions_temp = np.random.rand(N)

    # BOT always strong opinion
    opinions_temp[bot] = 1.0

    # Copy influence matrix
    W_temp = W.copy()

    # Pick influencer based on centrality
    influencer = max(centrality_values, key=centrality_values.get)

    # Strengthen influencer effect
    for i in range(N):

        if i != influencer:

            W_temp[i][influencer] += 0.5

            # Normalize row
            W_temp[i] = W_temp[i] / np.sum(W_temp[i])

    # Track convergence
    disagreement = []

    for t in range(30):

        # Update opinions
        opinions_temp = W_temp @ opinions_temp

        # Bot opinion stays fixed
        opinions_temp[bot] = 1.0

        # Measure disagreement in network
        disagreement.append(np.std(opinions_temp))

    # Save results
    convergence_history[method_name] = disagreement

In [ ]:
plt.figure(figsize=(10,6))

for method, values in convergence_history.items():
    plt.plot(values, label=method)

plt.title("Convergence Under Bot Influence (Different Centralities)")
plt.xlabel("Time Steps")
plt.ylabel("Opinion Spread (Std Dev)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
df = pd.DataFrame({
    "FinalOpinion": opinions,
    "Degree": list(degree.values()),
    "Betweenness": list(betweenness.values()),
    "Closeness": list(closeness.values())
})

print(df.corr()["FinalOpinion"])

In [ ]:
def run_simulation_full(centrality_values, W, N, bot, steps=30):

    opinions = np.random.rand(N)
    opinions[bot] = 1.0

    influencer = max(centrality_values, key=centrality_values.get)

    W_temp = W.copy()

    for i in range(N):
        if i != influencer:
            W_temp[i][influencer] += 0.5
            W_temp[i] = W_temp[i] / np.sum(W_temp[i])

    history = []

    for t in range(steps):

        opinions = W_temp @ opinions
        opinions[bot] = 1.0

        history.append(opinions.copy())

    return np.array(history), influencer

In [ ]:
bot = 1

deg_hist, deg_inf = run_simulation_full(degree, W, N, bot)
bet_hist, bet_inf = run_simulation_full(betweenness, W, N, bot)
clo_hist, clo_inf = run_simulation_full(closeness, W, N, bot)

In [ ]:
def plot_convergence(history, title, bot):

    plt.figure(figsize=(10,5))

    for i in range(N):

        if i == bot:
            plt.plot(history[:, i], color="red", linewidth=3, label="Bot")
        else:
            plt.plot(history[:, i], alpha=0.4)

    plt.title(title)
    plt.xlabel("Time Steps")
    plt.ylabel("Opinion Value")
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
plot_convergence(
    deg_hist,
    "Opinion Convergence (Degree Centrality + Bot)",
    bot
)

In [ ]:
plot_convergence(
    bet_hist,
    "Opinion Convergence (Betweenness Centrality + Bot)",
    bot
)

In [ ]:
plot_convergence(
    clo_hist,
    "Opinion Convergence (Closeness Centrality + Bot)",
    bot
)


In [ ]:
steps = 30
history = [opinions.copy()]

for t in range(steps):

    # Normal opinion update
    opinions = W @ opinions

    # Strong persistent bot influence
    opinions = 0.7 * opinions + 0.3 * opinions[bot]

    # Bot remains fixed
    opinions[bot] = 1.0

    # Store history
    history.append(opinions.copy())

In [ ]:
final_order = np.argsort(history[-1])

In [ ]:
import seaborn as sns
import matplotlib.patches as mpatches

# Get the final opinions and sort
final_opinions = history[-1]
final_order = np.argsort(final_opinions)

# Reorder the history array
sorted_history = history[:, final_order].T

# Create a custom colormap that highlights the bot
# First, create a copy of the data
highlighted_data = sorted_history.copy()

# Create the figure
fig, ax = plt.subplots(figsize=(14, 10))

# Create the heatmap
sns.heatmap(
    sorted_history,
    cmap='coolwarm',
    center=0.5,
    cbar_kws={'label': 'Opinion Value (0 to 1)'},
    xticklabels=5,
    yticklabels=False,
    vmin=0,
    vmax=1,
    ax=ax
)

# Find the bot's position and highlight the row
bot_position = np.where(final_order == bot)[0][0]

# Add a rectangle to highlight the bot's row
ax.axhline(
    y=bot_position,
    color='gold',
    linestyle='-',
    linewidth=3,
    alpha=0.8
)

# Add a label for the bot
ax.text(
    -0.5,  # x position (left of heatmap)
    bot_position + 0.5,  # y position (aligned with bot row)
    ' BOT →',
    fontsize=12,
    fontweight='bold',
    color='gold'
)

plt.title('Opinion Dynamics (Sorted by Final Opinion) - Bot Highlighted in Gold')
plt.xlabel('Time Steps')
plt.ylabel('Individuals (Sorted from Lowest to Highest Final Opinion)')

# Add custom legend
legend_elements = [
    mpatches.Patch(color='gold', label='Bot Row'),
    mpatches.Patch(color='red', label='High Opinion (1.0)'),
    mpatches.Patch(color='blue', label='Low Opinion (0.0)')
]
plt.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.set_ylim(0, 1)
ax.set_xlim(0, N)

bars = ax.bar(range(N), history[0])

def update(frame):
    for i, b in enumerate(bars):
        b.set_height(history[frame][i])
    ax.set_title(f"Opinion Spread Step {frame}")
    return bars

ani = animation.FuncAnimation(fig, update, frames=len(history), interval=400)

HTML(ani.to_jshtml())